In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


## Update the repo


In [ ]:

import os

PROJECT = str(ROOT)
REPO = f'{PROJECT}/OneTrainer'

os.makedirs(PROJECT, exist_ok=True)

if not os.path.exists(REPO):
    %cd $PROJECT
    !git clone --recursive https://github.com/Nerogar/OneTrainer.git

%cd $REPO
!rm -rf /content/OneTrainer /content/diffusers

!git log -1 --format="OneTrainer commit: %h  %cd"

print("cwd:", os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
OneTrainer commit: 9a270603  Sun Feb 8 09:18:39 2026 +0100
Active dir: /content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer


## Install requirements


In [ ]:
# exact versions this commit wants. restart the runtime after, re-run the
# mount cell
%cd {ROOT}/OneTrainer
!pip install -r requirements.txt


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Obtaining diffusers from git+https://github.com/huggingface/diffusers.git@6a1904e#egg=diffusers (from -r requirements-global.txt (line 23))
  Updating ./src/diffusers clone (to revision 6a1904e)
  Running command git fetch -q --tags
  Running command git reset --hard -q 6a1904e
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
Obtaining mgds from git+https://github.com/Nerogar/mgds.git@a0c84a3#egg=mgds (from -r requirements-global.txt (line 35))
  Updating ./src/mgds clone (to revision a0c84a3)
  Running command git fetch -q --tags
  Running command git reset --hard -q a0c84a3
  Installing build dependen

In [ ]:
!python -c "import torch, torchvision, transformers, diffusers; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('transformers', transformers.__version__); print('diffusers', diffusers.__version__)"
!python -c "import torch; print('cuda:', torch.cuda.is_available())"
!python -c "from torchvision.io import write_video; print('write_video OK')"
!python -c "import mgds; print('mgds:', mgds.__file__)"


torch 2.8.0+cu128
torchvision 0.23.0+cu128
transformers 4.56.2
diffusers 0.37.0.dev0
cuda: True
write_video OK
mgds: None


## Check the config


In [ ]:
# catches the failures that give a silent no-op run: empty concepts list,
# paths resolving somewhere else, missing image folders, tensorboard on
import json, os

CONFIG = str(ROOT / "pixartsigma_colab.json")
REPO = str(ROOT / "OneTrainer")

c = json.load(open(CONFIG))

for k in ["base_model_name", "model_type", "training_method", "peft_type",
          "train_device", "temp_device", "resolution", "epochs", "batch_size",
          "learning_rate", "save_every", "save_every_unit",
          "workspace_dir", "output_model_destination"]:
    print(f"{k}: {c.get(k)}")

print()
problems = []

if c.get("concepts") == []:
    problems.append("concepts is [] -- must be null, or the concept file is ignored "
                    "and training silently runs on zero images")
if c.get("train_device") != "cuda":
    problems.append(f"train_device is {c.get('train_device')!r} -- should be 'cuda'")
if c.get("tensorboard"):
    problems.append("tensorboard is true -- Colab has no /usr/bin/tensorboard, set false")

cf = c.get("concept_file_name")
if cf:
    resolved = cf if os.path.isabs(cf) else os.path.join(REPO, cf)
    print("concept file:", resolved, "| exists:", os.path.exists(resolved))
    if os.path.exists(resolved):
        for con in json.load(open(resolved)):
            d = con.get("path")
            n = len(os.listdir(d)) if d and os.path.isdir(d) else 0
            print(f"  concept path: {d} | files: {n}")
            if n == 0:
                problems.append(f"concept path contains no paths: {d}")
    else:
        problems.append("The concept file was not found")

print()
if problems:
    for p in problems:
        print("problem:", p)
else:
    print("config Ok")


base_model_name: PixArt-alpha/PixArt-Sigma-XL-2-1024-MS
model_type: PIXART_SIGMA
training_method: LORA
peft_type: LORA
train_device: cuda
temp_device: cpu
resolution: 1024
epochs: 100
batch_size: 4
learning_rate: 0.0001
save_every: 0
save_every_unit: NEVER
workspace_dir: /content/drive/MyDrive/Synthetic_Plants_Project/workspace/pixart_achillea_run
output_model_destination: /content/drive/MyDrive/Synthetic_Plants_Project/outputs/pixart_achillea/lora.safetensors

concept file: /content/drive/MyDrive/Synthetic_Plants_Project/Notebooks/OneTrainer/modelconfigs/train_concepts.json | exists: True
  concept path: /content/drive/MyDrive/Synthetic_Plants_Project/Datasets/Achillea_Maritima_2 | files: 276

config looks OK


## Train


In [ ]:
# the shim only patches torchvision.io.write_video, which some builds no
# longer export
%%writefile /content/shim.py
import sys, os, runpy
import torchvision.io
if not hasattr(torchvision.io, "write_video"):
    torchvision.io.write_video = lambda *a, **k: None

root = os.getcwd()
sys.path.insert(0, os.path.join(root, "scripts"))
sys.path.insert(0, root)

sys.argv = sys.argv[1:]
runpy.run_path(os.path.join(root, "scripts", "train.py"), run_name="__main__")


Overwriting /content/shim.py


In [ ]:
!find {ROOT} -name "*.jpg" -newermt "-1 hour" 2>/dev/null | head -20
!find /content -name "*.jpg" -newermt "-1 hour" -not -path "*/drive/*" 2>/dev/null | head -20


In [ ]:
!ls -la {ROOT}/training/concepts
!ls -la {ROOT}/training/concepts/pixart/ 2>/dev/null
!ls -la {ROOT}/training/concepts/pixart


total 31
drwx------ 2 root root 4096 Aug 25 16:59 flux_lora
-rw------- 1 root root 9427 Aug 26 20:51 generate_eval.py
-rw------- 1 root root 5170 Aug 26 19:33 Label.ipynb
drwx------ 2 root root 4096 Aug 25 16:59 pixart_lora
drwx------ 2 root root 4096 Aug 25 16:58 qwen_lora
drwx------ 2 root root 4096 Aug 25 16:59 sdxl_lora
total 56
-rw------- 1 root root 14980 Aug 25 16:56 pixartsigma_colab_achillea.json
-rw------- 1 root root 14995 Aug 25 16:56 pixartsigma_colab_carpobrotus.json
-rw------- 1 root root 14980 Aug 25 16:56 pixartsigma_colab_eryngium.json
-rw------- 1 root root  2254 Aug 25 16:56 train_concepts_achillea.json
-rw------- 1 root root  2270 Aug 25 16:56 train_concepts_carpobrotus.json
-rw------- 1 root root  2256 Aug 25 16:56 train_concepts_eryngium.json
-rw------- 1 root root   832 Aug 25 16:56 train_samples_achillea.json
-rw------- 1 root root   832 Aug 25 16:56 train_samples_carpobrotus.json
-rw------- 1 root root   832 Aug 25 16:56 train_samples_eryngium.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/pixart/pixartsigma_colab_achillea.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/pixart/pixartsigma_colab_eryngium.json


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
2026-08-26 22:13:44.046850: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 22:13:44.119132: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Clearing cache di

In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/pixart/pixartsigma_colab_carpobrotus.json


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
2026-08-26 23:04:49.179224: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 23:04:49.250689: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Clearing cache di

In [ ]:
%cd {ROOT}
!python Configs_and_Concepts/generate_eval.py --all --n 100


In [ ]:
# @title
%cd {ROOT}
!python Configs_and_Concepts/generate_eval.py --species achillea --n 2


/content/drive/MyDrive/Synthetic_Plants_Project
2026-08-26 20:52:04.171909: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 20:52:04.243580: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
generating 2 per species: ac

In [ ]:
# @title
from safetensors import safe_open
p = str(ROOT / "outputs/lora/pixart_achillea/lora.safetensors")
with safe_open(p, framework="pt") as f:
    keys = list(f.keys())
print(len(keys), "tensors")
for k in keys[:12]:
    print(" ", k)


861 tensors
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_up.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_up.weight
  lora_transformer_adaln_single_linear.alpha
  lora_transformer_adaln_single_linear.lora_down.weight
  lora_transformer_adaln_single_linear.lora_up.weight
  lora_transformer_caption_projection_linear_1.alpha
  lora_transformer_caption_projection_linear_1.lora_down.weight
  lora_transformer_caption_projection_linear_1.lora_up.weight


## Backup loading


In [ ]:
# @title
# !ls -la {ROOT}/outputs/workspace*/backup/

# %cd {ROOT}/OneTrainer
# !python -u /content/shim.py train.py \
#   --config-path {ROOT}/training/configs/pixart/pixartsigma_colab.json \
#   --resume-from-checkpoint {ROOT}/outputs/workspace<run>/backup/last


In [ ]:
# @title
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
!pip freeze > {ROOT}/requirements_frozen_{stamp}.txt
print("written: requirements_frozen_" + stamp + ".txt")


written: requirements_frozen_20260825_0018.txt
